# Timeouts & Retries

This notebook covers `ModelSettings(timeout=...)` and `ModelRetrySettings`.


## Model-call timeouts

`ModelSettings(timeout=...)` caps how long a single model call may take —
simulated with a scripted responder that sleeps longer than the timeout.


In [1]:
import asyncio  # lets us fake a slow model with a pause

from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    ModelSettings,  # settings for how the model behaves, like effort, timeout, and retries
    ModelTimeoutError,  # error raised when one model call takes too long
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ModelStep,  # one scripted step, made from your own function
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
)


# A step function that acts like a slow model call. It is used in
# ScriptedModel([ModelStep(responder=slow_responder)]) below.
# responder= means this function runs when the model is called, so it can pause first.
# A fixed ModelStep(output=[...]) could not pause. It would answer right away
async def slow_responder(call) -> ModelStep:  # call: the model request that is running now
    """A scripted responder that takes 0.3s to answer — used to trigger a timeout."""
    await asyncio.sleep(0.3)
    return ModelStep(output=[assistant_message("finally answered")])


# Demo: sets the timeout much shorter than the pause in slow_responder
# and shows the run stops with ModelTimeoutError instead of waiting
async def demo_timeout() -> None:
    # Initialize the agent
    timeout_agent = Agent(
        name="timeout-demo",
        instructions="test",
        # model_settings: the timeout is much shorter than the 0.3 second pause
        model_settings=ModelSettings(timeout=0.05),
    )

    # Run the Agent
    try:
        await Runner.run(
            timeout_agent,
            "x",
            # run_config: settings for just this run
            run_config=RunConfig(
                # model: use this model for this run only
                model=ScriptedModel(
                    [
                        ModelStep(responder=slow_responder),
                    ]
                ),
                # workflow_name: the name this run shows up under in tracing
                workflow_name="e07-timeout",
            ),
        )
    except ModelTimeoutError as error:
        # proves a model call that takes longer than model_settings.timeout is stopped with
        # ModelTimeoutError and is not allowed to finish
        print("[expected]", error)


await demo_timeout()
# Outside a notebook: asyncio.run(demo_timeout())

[expected] Model call timed out after 0.05 seconds.


## Retries — a real gotcha

Confirmed by reading the SDK's own retry loop: `ModelRetrySettings(max_retries=N)`
alone retries nothing — a `policy=` (e.g.
`agents.retry.retry_policies.network_error()`) is also required.

`max_retries` is easy to mix up with `max_turns`. They count different things:

|                          | `max_turns`                            | `max_retries`                                  |
| ------------------------ | -------------------------------------- | ---------------------------------------------- |
| Where it's set           | `Runner.run(...)`                      | `ModelSettings(retry=ModelRetrySettings(...))` |
| What it counts           | Loop iterations (model call + actions) | Re-tries of one failed model call              |
| Triggered by             | Normal agent progress                  | A model-call error                             |
| Needs another setting    | No                                     | Yes, a `policy=`                               |
| Hitting the limit raises | `MaxTurnsExceeded`                     | The original error (e.g. `ModelTimeoutError`)  |


In [4]:
import agents.retry as retry_module  # ready-made retry rules

from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    ModelRetrySettings,  # says if and how a failed model call is tried again
    ModelSettings,  # settings for how the model behaves, like effort, timeout, and retries
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ModelStep,  # one scripted step, made from your own function
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
)

call_count = 0


# A step function that fakes a short network problem on the first call.
# It is used in ScriptedModel([ModelStep(responder=flaky_responder), ...]) below.
# in that slot, and it either raises an error or returns a fixed ModelStep(output=...)
async def flaky_responder(call) -> ModelStep:  # call: the model request that is running now
    """Fails once with a retryable error, then succeeds — proves whether a retry happened."""
    global call_count
    call_count += 1
    if call_count == 1:
        # Fail on the first call: raising here makes the slot act like a broken model call
        raise TimeoutError("simulated transient network error")
    # Work on later calls: this inner ModelStep is the fixed answer the slot gives back
    # ModelStep(output=[...])      -> a fixed answer, always the same, so it can never fail
    return ModelStep(output=[assistant_message("succeeded after retry")])

In [5]:
# Demo: the first of two scripted retry demos. Only max_retries is set (no policy=), so the
# failure is not tried again and the run stops right away
async def demo_retries_no_policy() -> None:
    global call_count

    # No policy: the error stops the run right away and nothing is tried again.
    call_count = 0
    # Initialize the no-policy agent
    no_policy_agent = Agent(
        name="retry-demo",
        instructions="test",
        # model_settings: only max_retries, and policy= is left out on purpose
        model_settings=ModelSettings(retry=ModelRetrySettings(max_retries=2)),
    )

    # Run the no-policy agent
    try:
        await Runner.run(
            no_policy_agent,
            "x",
            # run_config: settings for just this run
            run_config=RunConfig(
                # model: use this model for this run only
                model=ScriptedModel(
                    [
                        # one slot only: with no policy there is no second try, so a second slot would go unused.
                        # responder= runs flaky_responder when this slot is reached, and it raises the error
                        # ModelStep(responder=func)    -> func runs when the model is called, and it picks the result
                        ModelStep(responder=flaky_responder),
                    ]
                ),
                # workflow_name: the name this run shows up under in tracing
                workflow_name="e07-retry-no-policy",
            ),
        )
    except TimeoutError as error:
        # proves max_retries alone (no policy=) never retries:
        # call_count stays at 1 and the error stops the run right away
        print(f"[no policy] call_count={call_count}, raised immediately: {error}")


await demo_retries_no_policy()
# Outside a notebook: asyncio.run(demo_retries_no_policy())

[no policy] call_count=1, raised immediately: simulated transient network error


In [6]:
# Demo: the second scripted retry demo. The same failure as above, but now policy= is added,
# so the SDK tries the model call again and the run works
async def demo_retries_with_policy() -> None:
    global call_count

    # With a policy: the same failure is now tried again.
    call_count = 0
    # Initialize the with-policy agent
    with_policy_agent = Agent(
        name="retry-demo",
        instructions="test",
        # model_settings: policy= is what turns retries on
        model_settings=ModelSettings(
            # retry: the rules for trying again when a model call fails
            retry=ModelRetrySettings(
                # max_retries: how many extra tries are allowed after the first one fails
                max_retries=2,
                # policy: the rule for when to retry. Without it, nothing is retried
                policy=retry_module.retry_policies.network_error(),
            )
        ),
    )

    # Run the with-policy agent
    result = await Runner.run(
        with_policy_agent,
        "x",
        # run_config: settings for just this run
        run_config=RunConfig(
            # model: use this model for this run only
            model=ScriptedModel(
                [
                    # two slots: one for the first try that fails, one for the retry that works.
                    # each slot runs flaky_responder again: the first raises, the second gives a fixed answer
                    ModelStep(responder=flaky_responder),
                    ModelStep(responder=flaky_responder),
                ]
            ),
            # workflow_name: the name this run shows up under in tracing
            workflow_name="e07-retry-with-policy",
        ),
    )
    # proves the same failure is tried again once (call_count reaches 2) and the
    # run still works, using the second scripted step as final_output
    print(f"[with policy] call_count={call_count}, final_output={result.final_output!r}")


await demo_retries_with_policy()
# Outside a notebook: asyncio.run(demo_retries_with_policy())

[with policy] call_count=2, final_output='succeeded after retry'


## Run it live — optional, needs a real key

Four real calls that mirror `demo_timeout()`, `demo_retries_no_policy()`, and
`demo_retries_with_policy()` above, with a
live model in place of the scripted `slow_responder`/`flaky_responder`. A
`timeout=0.01` is shorter than any real round trip to the API, so the first
three calls are fully deterministic. The first proves a real call gets cut off
with `ModelTimeoutError`. The second adds `max_retries=2` with no `policy=`, and
the call still fails after one attempt. The third wraps
`retry_policies.network_error()` in a small counting policy. A
`ModelTimeoutError` counts as a timeout for that policy (confirmed in the SDK's
`run_internal/model_retry.py`), so the log shows the SDK really retrying twice
before giving up. The fourth keeps the same retry settings with a realistic
timeout and simply succeeds, which shows retries don't get in the way of a
healthy call.


In [8]:
import os  # reads OPENAI_API_KEY from the environment

import agents.retry as retry_module  # ready-made retry rules

from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    ModelRetrySettings,  # says if and how a failed model call is tried again
    ModelSettings,  # settings for how the model behaves, like effort, timeout, and retries
    ModelTimeoutError,  # error raised when one model call takes too long
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.retry import (
    RetryPolicyContext,  # info given to a retry rule: the error, which try this is, and the max tries
)
from dotenv import load_dotenv  # loads the .env file into the environment

load_dotenv()

# Saves every attempt number the counting policy below sees, so we can
# see the retries instead of them happening quietly inside the SDK
retry_log: list[int] = []

network_error_policy = retry_module.retry_policies.network_error()


# A retry rule that wraps network_error() and writes down each retry choice. It is used in
# ModelRetrySettings(policy=counting_network_error_policy) on the live with-policy and healthy agents below
def counting_network_error_policy(context: RetryPolicyContext) -> bool:
    retry_log.append(context.attempt)
    return network_error_policy(context)

In [9]:
# Demo: the first of four live calls, like demo_timeout() above but with the real model.
# A 0.01 second timeout is shorter than any real call, so it always ends in ModelTimeoutError
async def demo_live_timeout() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    # Initialize the timeout agent
    timeout_agent = Agent(
        name="Leaf Village Assistant",
        instructions="Answer briefly.",
        # model_settings: 0.01 seconds is much shorter than any real API call
        model_settings=ModelSettings(timeout=0.01),
    )

    # Run the timeout agent
    try:
        await Runner.run(
            timeout_agent,
            "What is a Kage-level ninja?",
            # run_config: settings for just this run
            run_config=RunConfig(workflow_name="e07-live-timeout"),
        )
    except ModelTimeoutError as error:
        # proves model_settings.timeout stops a real API call that is still running with
        # ModelTimeoutError, not only the fake slow_responder above
        print("[live timeout]", error)


await demo_live_timeout()
# Outside a notebook: asyncio.run(demo_live_timeout())

[live timeout] Model call timed out after 0.01 seconds.


In [10]:
# Demo: the live version of demo_retries_no_policy(). max_retries=2 is set but policy= is
# left out, so the first timeout stops the run
async def demo_live_no_policy() -> None:
    # Initialize the no-policy agent
    no_policy_agent = Agent(
        name="Leaf Village Assistant",
        instructions="Answer briefly.",
        # model_settings: only max_retries, and policy= is left out on purpose
        model_settings=ModelSettings(
            # timeout: how many seconds to wait for the model before giving up
            timeout=0.01,
            # retry: the rules for trying again when a model call fails
            retry=ModelRetrySettings(max_retries=2),
        ),
    )

    # Run the no-policy agent
    try:
        await Runner.run(
            no_policy_agent,
            "What is a Kage-level ninja?",
            # run_config: settings for just this run
            run_config=RunConfig(workflow_name="e07-live-retry-no-policy"),
        )
    except ModelTimeoutError as error:
        # proves max_retries alone never retries a real failure. With no policy, the SDK
        # has nothing to ask, so the first timeout stops the run right away
        print("[live no policy] raised after one attempt:", error)


await demo_live_no_policy()
# Outside a notebook: asyncio.run(demo_live_no_policy())

[live no policy] raised after one attempt: Model call timed out after 0.01 seconds.


In [11]:
# Demo: the live version of demo_retries_with_policy(). The counting policy is added, so the SDK
# asks it after each timeout and tries again until max_retries runs out
async def demo_live_with_policy() -> None:
    retry_log.clear()
    # Initialize the with-policy agent
    with_policy_agent = Agent(
        name="Leaf Village Assistant",
        instructions="Answer briefly.",
        # model_settings: policy= is what turns retries on
        model_settings=ModelSettings(
            # timeout: how many seconds to wait for the model before giving up
            timeout=0.01,
            # retry: the rules for trying again when a model call fails
            retry=ModelRetrySettings(
                # max_retries: how many extra tries are allowed after the first one fails
                max_retries=2,
                # policy: the rule for when to retry. Without it, nothing is retried
                policy=counting_network_error_policy,
            ),
        ),
    )

    # Run the with-policy agent
    try:
        await Runner.run(
            with_policy_agent,
            "What is a Kage-level ninja?",
            # run_config: settings for just this run
            run_config=RunConfig(workflow_name="e07-live-retry-with-policy"),
        )
    except ModelTimeoutError as error:
        # proves that with policy= the SDK asks the policy after each real timeout and
        # tries again. The log has one entry for each retry choice before it gives up
        print(f"[live with policy] retry decisions for attempts {retry_log}, then raised: {error}")


await demo_live_with_policy()
# Outside a notebook: asyncio.run(demo_live_with_policy())

[live with policy] retry decisions for attempts [1, 2], then raised: Model call timed out after 0.01 seconds.


In [12]:
# Demo: a healthy call with a normal timeout and the same retry settings. It works on the
# first try, so the policy is never asked
async def demo_live_with_healthy() -> None:
    retry_log.clear()
    # Initialize the healthy agent
    healthy_agent = Agent(
        name="Leaf Village Assistant",
        instructions="Answer briefly.",
        # model_settings: a normal timeout, with the same retry policy as above
        model_settings=ModelSettings(
            # timeout: how many seconds to wait for the model before giving up
            timeout=60,
            # retry: the rules for trying again when a model call fails
            retry=ModelRetrySettings(
                # max_retries: how many extra tries are allowed after the first one fails
                max_retries=2,
                # policy: the rule for when to retry. Without it, nothing is retried
                policy=counting_network_error_policy,
            ),
        ),
    )

    # Run the healthy agent
    healthy_result = await Runner.run(
        healthy_agent,
        "What is a Kage-level ninja?",
        # run_config: settings for just this run
        run_config=RunConfig(workflow_name="e07-live-healthy"),
    )
    # proves a healthy call with the same retry settings never asks the policy
    # (the retry log is empty) and gives back a normal final_output
    print(f"[live healthy] retry decisions {retry_log}, final_output={healthy_result.final_output!r}")


await demo_live_with_healthy()
# Outside a notebook: asyncio.run(demo_live_with_healthy())

[live healthy] retry decisions [], final_output='A **Kage-level ninja** is an exceptionally powerful shinobi in *Naruto*, considered capable of fighting or rivaling a village leader—the **Kage** (such as Hokage, Kazekage, or Mizukage).\n\nIt’s an informal power-scaling term, not an official rank. Kage-level ninja typically possess overwhelming skill, advanced techniques, great battle experience, or unusually powerful abilities.'


## Summary

- `ModelSettings(timeout=...)` caps how long a single model call may
  take, raising `ModelTimeoutError` if it's exceeded.
- `ModelRetrySettings(max_retries=...)` does nothing on its own — a
  `policy=` (e.g. `agents.retry.retry_policies.network_error()`) is also
  required before a failed model call actually gets retried.
